# ReadyAI CHiRPE SmolLM2 Summarization + Segmented Inference

This notebook is the **lightweight** sibling of
`05_readyai_onnx_summarization_inference.ipynb`. The pipeline and responsibility
boundary are identical — the entire flow runs **inside ReadyAI** — but the
summarization stage swaps the Phi-3 ONNX summarizer for
[`SmolLM2`](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct), a small
(360M-parameter by default) instruction-tuned causal LM served through plain
`transformers`.

Why SmolLM2:

- **Light.** 360M params vs. Phi-3-mini's 3.8B. It loads and runs on CPU with a
  small memory footprint, making it a good default for constrained ReadyAI hosts.
- **Standard dependency.** It uses `transformers` directly (a core CHiRPE
  dependency), so no `onnxruntime-genai` / `onnx-llm` extra is required.
- **Same contract.** `SmolLM2Summarizer` produces the same single short,
  third-person clinical summary per segment, with **greedy deterministic
  decoding** (`do_sample=False`), so it is a drop-in alternative.

Raw transcript JSON goes in, and the aggregated CHR-P decision comes out — all
within ReadyAI.

## Changelog

- 2026-06-18: Added SmolLM2 ReadyAI summarization notebook mirroring notebook 05; evaluates a lightweight transformers-based summarizer inside the same ReadyAI pipeline.
- 2026-06-18: Replaced Phi-3 ONNX summarization with `SmolLM2Summarizer`; reduces dependency and memory requirements by avoiding `onnxruntime-genai` / `onnx-llm`.
- 2026-06-18: IMPORTANT: Preserved the newer `segment text + summary` classifier input, not the older summary-only input, and kept transcript aggregation as average-probability voting; isolates the intended change to the summarizer swap while preserving downstream behavior.

## Responsibility Boundary

The entire pipeline runs **inside ReadyAI**.

| Step | Location |
| --- | --- |
| Raw transcript ingestion | Inside ReadyAI |
| Raw transcript segmentation | Inside ReadyAI |
| **Segment summarization (SmolLM2)** | **Inside ReadyAI** |
| CHiRPE model inference on each segment | Inside ReadyAI |
| Transcript-level voting or aggregation | Inside ReadyAI |

## Inside ReadyAI: Raw Transcript Input

The payload below is the raw transcript that ReadyAI ingests. ReadyAI owns every
step from here on.

In [1]:
raw_transcript_payload = {
    "participant_id": "demo_001",
    "transcript": [
        {"speaker": "interviewer", "text": "Tell me about whether events have had special meaning for you."},
        {"speaker": "interviewee", "text": "Sometimes I feel like television messages are personally directed at me."},
        {"speaker": "interviewer", "text": "Tell me whether people are talking about you or watching you."},
        {"speaker": "interviewee", "text": "At times I worry strangers are watching me when I am outside, but it comes and goes."},
        {"speaker": "interviewer", "text": "Tell me about any trouble concentrating or focusing."},
        {"speaker": "interviewee", "text": "I have mild trouble focusing at school, but I do not get lost or confused about where I am."},
    ],
}

raw_transcript_payload

{'participant_id': 'demo_001',
 'transcript': [{'speaker': 'interviewer',
   'text': 'Tell me about whether events have had special meaning for you.'},
  {'speaker': 'interviewee',
   'text': 'Sometimes I feel like television messages are personally directed at me.'},
  {'speaker': 'interviewer',
   'text': 'Tell me whether people are talking about you or watching you.'},
  {'speaker': 'interviewee',
   'text': 'At times I worry strangers are watching me when I am outside, but it comes and goes.'},
  {'speaker': 'interviewer',
   'text': 'Tell me about any trouble concentrating or focusing.'},
  {'speaker': 'interviewee',
   'text': 'I have mild trouble focusing at school, but I do not get lost or confused about where I am.'}]}

## Inside ReadyAI: Segment the Raw Transcript

This cell performs **segmentation inside ReadyAI** using `SymptomSegmenter`.
Unmapped segments are dropped, mirroring `TranscriptPreprocessor`.

In [2]:
from chirpe.data.segmentation import SymptomSegmenter

segmenter = SymptomSegmenter(threshold=0.80)
segments = segmenter.segment_transcript(raw_transcript_payload["transcript"])

mapped_segments = [seg for seg in segments if seg.domain != "unmapped"]
if not mapped_segments:
    raise ValueError("Segmentation produced no mapped segments; check the prompts and threshold.")

participant_id = raw_transcript_payload["participant_id"]
segmented_payload = {
    "participant_id": participant_id,
    "segments": [
        {
            "segment_id": f"{participant_id}_seg_{index + 1:03d}",
            "domain": segment.domain,
            "text": segment.get_text(),
            "start_idx": segment.start_idx,
            "end_idx": segment.end_idx,
        }
        for index, segment in enumerate(mapped_segments)
    ],
}

import json
print(json.dumps(segmented_payload, indent=2))

{
  "participant_id": "demo_001",
  "segments": [
    {
      "segment_id": "demo_001_seg_001",
      "domain": "P1_Unusual_Thoughts",
      "text": "Tell me about whether events have had special meaning for you. Sometimes I feel like television messages are personally directed at me.",
      "start_idx": 0,
      "end_idx": 1
    },
    {
      "segment_id": "demo_001_seg_002",
      "domain": "P2_Suspiciousness",
      "text": "Tell me whether people are talking about you or watching you. At times I worry strangers are watching me when I am outside, but it comes and goes.",
      "start_idx": 2,
      "end_idx": 3
    },
    {
      "segment_id": "demo_001_seg_003",
      "domain": "P4_Disoriented",
      "text": "Tell me about any trouble concentrating or focusing. I have mild trouble focusing at school, but I do not get lost or confused about where I am.",
      "start_idx": 4,
      "end_idx": 5
    }
  ]
}


## Inside ReadyAI: Load the SmolLM2 Summarizer

ReadyAI loads the lightweight `SmolLM2Summarizer`. The model is loaded once and
reused across segments. The first run downloads the checkpoint from the Hugging
Face Hub (~360M params for the default `SmolLM2-360M-Instruct`); set
`SMOLLM2_MODEL_NAME` to `HuggingFaceTB/SmolLM2-135M-Instruct` for an even smaller
footprint or `...-1.7B-Instruct` for higher quality.

**Generation settings.** Decoding is **greedy and deterministic** —
`do_sample=False` with `num_beams=1`, pinned inside
`SmolLM2Summarizer.summarize_segment`. `max_new_tokens` is the only length
control.

In [3]:
from chirpe.data.summarizer import SmolLM2Summarizer

SMOLLM2_MODEL_NAME = SmolLM2Summarizer.DEFAULT_MODEL_NAME  # "HuggingFaceTB/SmolLM2-360M-Instruct"
SMOLLM2_DEVICE = "cpu"   # "auto"/"cuda" if a GPU is available
SMOLLM2_MAX_NEW_TOKENS = 64

summarizer = SmolLM2Summarizer(
    model_name=SMOLLM2_MODEL_NAME,
    device=SMOLLM2_DEVICE,
    max_new_tokens=SMOLLM2_MAX_NEW_TOKENS,
)

print(f"Loaded SmolLM2 summarizer: {SMOLLM2_MODEL_NAME} (device={SMOLLM2_DEVICE})")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded SmolLM2 summarizer: HuggingFaceTB/SmolLM2-360M-Instruct (device=cpu)


In [4]:
# Generation settings used for every segment, pinned in
# SmolLM2Summarizer.summarize_segment. Decoding is greedy and deterministic.
generation_settings = {
    "do_sample": False,   # greedy decoding (deterministic)
    "num_beams": 1,       # no beam search
    "max_new_tokens": summarizer.max_new_tokens,
}
generation_settings

{'do_sample': False, 'num_beams': 1, 'max_new_tokens': 64}

## Inside ReadyAI: Summarize Each Segment

ReadyAI summarizes each segment's raw `text` with SmolLM2, producing one short
third-person clinical summary per segment.

In [5]:
summaries = []
for segment in segmented_payload["segments"]:
    summary = summarizer.summarize_segment(segment["text"])
    summaries.append(summary)
    print(f"[{segment['segment_id']}] {segment['domain']}")
    print(f"  raw    : {segment['text']}")
    print(f"  summary: {summary}\n")

[demo_001_seg_001] P1_Unusual_Thoughts
  raw    : Tell me about whether events have had special meaning for you. Sometimes I feel like television messages are personally directed at me.
  summary: You often feel like television messages are directed at you, and it can be difficult to distinguish between what's real and what's not.



[demo_001_seg_002] P2_Suspiciousness
  raw    : Tell me whether people are talking about you or watching you. At times I worry strangers are watching me when I am outside, but it comes and goes.
  summary: At times, I worry strangers are watching me when I am outside, but it comes and goes.



[demo_001_seg_003] P4_Disoriented
  raw    : Tell me about any trouble concentrating or focusing. I have mild trouble focusing at school, but I do not get lost or confused about where I am.
  summary: Mild trouble concentrating or focusing at school, but not getting lost or confused about where you are.



## Inside ReadyAI: Load CHiRPE Model

ReadyAI loads the CHiRPE classifier. Change `MODEL_PATH` to test a different
checkpoint. This step requires a trained `best_model` directory under
`outputs/` — see `AGENTS.md` / the training console scripts to produce one.

In [6]:
from pathlib import Path

from chirpe.models.classifier import CHRClassifier

# Prefer the classifier trained on segment+summary input; fall back to the older
# summary-only checkpoint if the combined one is not present.
MODEL_PATH_CANDIDATES = [
    Path("outputs/string_onnx_checkpoints_combined/bert/best_model"),
    Path("../outputs/string_onnx_checkpoints_combined/bert/best_model"),
    Path("outputs/string_onnx_checkpoints/bert/best_model"),
    Path("../outputs/string_onnx_checkpoints/bert/best_model"),
]
MODEL_PATH = next((path for path in MODEL_PATH_CANDIDATES if path.exists()), MODEL_PATH_CANDIDATES[0])

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model checkpoint not found: {MODEL_PATH}. Update MODEL_PATH to a trained CHiRPE best_model directory."
    )

classifier = CHRClassifier(model_name=str(MODEL_PATH))
classifier.load(MODEL_PATH)

print(f"Loaded CHiRPE model from: {MODEL_PATH}")

Loaded CHiRPE model from: ../outputs/string_onnx_checkpoints_combined/bert/best_model


## Inside ReadyAI: Predict per Segment

ReadyAI runs `classifier.predict()` on **`segment text + summary`**
(`build_classifier_input`) to produce per-segment results. Feeding the raw
segment alongside the SmolLM2 summary makes the classifier robust to a poor or
hallucinated summary — the original wording is always present, so a bad summary
degrades rather than dominates the prediction.

In [7]:
from chirpe.data.preprocessor import build_classifier_input

LABELS = {0: "Healthy", 1: "CHR-P"}

classifier_inputs = [
    build_classifier_input(segment["text"], summary)
    for segment, summary in zip(segmented_payload["segments"], summaries)
]

predictions, probabilities = classifier.predict(classifier_inputs, return_probs=True)

segment_results = []
for segment, summary, pred, prob in zip(
    segmented_payload["segments"], summaries, predictions, probabilities
):
    segment_results.append(
        {
            "segment_id": segment["segment_id"],
            "domain": segment["domain"],
            "summary": summary,
            "prediction": int(pred),
            "label": LABELS[int(pred)],
            "probabilities": [float(p) for p in prob],
        }
    )

for result in segment_results:
    print(f"[{result['segment_id']}] {result['domain']}: {result['label']} (p={result['probabilities']})")

[demo_001_seg_001] P1_Unusual_Thoughts: CHR-P (p=[0.09353215247392654, 0.9064677953720093])
[demo_001_seg_002] P2_Suspiciousness: CHR-P (p=[0.09907513111829758, 0.9009248614311218])
[demo_001_seg_003] P4_Disoriented: CHR-P (p=[0.10232595354318619, 0.8976740837097168])


## Inside ReadyAI: Transcript-Level Aggregation

Finally, ReadyAI aggregates the per-segment predictions into a single
transcript-level CHR-P decision by **average-probability voting** over the
segments (the project's default aggregation).

In [8]:
import numpy as np

prob_matrix = np.array([result["probabilities"] for result in segment_results])
mean_probs = prob_matrix.mean(axis=0)
transcript_pred = int(mean_probs.argmax())

transcript_result = {
    "participant_id": segmented_payload["participant_id"],
    "summarizer": SMOLLM2_MODEL_NAME,
    "n_segments": len(segment_results),
    "mean_probabilities": [float(p) for p in mean_probs],
    "prediction": transcript_pred,
    "label": LABELS[transcript_pred],
    "segments": segment_results,
}

print(json.dumps(transcript_result, indent=2))

{
  "participant_id": "demo_001",
  "summarizer": "HuggingFaceTB/SmolLM2-360M-Instruct",
  "n_segments": 3,
  "mean_probabilities": [
    0.09831107904513676,
    0.9016889135042826
  ],
  "prediction": 1,
  "label": "CHR-P",
  "segments": [
    {
      "segment_id": "demo_001_seg_001",
      "domain": "P1_Unusual_Thoughts",
      "summary": "You often feel like television messages are directed at you, and it can be difficult to distinguish between what's real and what's not.",
      "prediction": 1,
      "label": "CHR-P",
      "probabilities": [
        0.09353215247392654,
        0.9064677953720093
      ]
    },
    {
      "segment_id": "demo_001_seg_002",
      "domain": "P2_Suspiciousness",
      "summary": "At times, I worry strangers are watching me when I am outside, but it comes and goes.",
      "prediction": 1,
      "label": "CHR-P",
      "probabilities": [
        0.09907513111829758,
        0.9009248614311218
      ]
    },
    {
      "segment_id": "demo_001_seg_00